# `cs336_basics/model.py` 复习笔记

这份笔记只讲 `cs336_basics/model.py` 里的函数，并且按照 **Transformer 真正执行时的数据流顺序** 来组织，而不是按照文件里的定义顺序。

目标不是只记住“代码怎么写”，而是复习时能回答下面这些问题：

- 这个函数在 Transformer 的哪一步出现？
- 它对应的数学公式是什么？
- 张量形状为什么要这样安排？
- 代码里为什么要用这个具体写法，而不是别的写法？

---

## 整体路线图

对这份作业代码来说，整条前向传播可以概括为：

$$
\text{token ids}
\xrightarrow{\text{embedding}}
X_0
\xrightarrow{\text{多个 Transformer blocks}}
X_L
\xrightarrow{\text{RMSNorm}}
\hat X_L
\xrightarrow{\text{lm head}}
\text{logits}
$$

其中每个 `transformer_block()` 的结构是：

$$
Y = X + \text{MHA}(\text{RMSNorm}(X))
$$

$$
Z = Y + \text{SwiGLU}(\text{RMSNorm}(Y))
$$

所以如果把所有函数放回模型里看，顺序大致是：

1. `embedding`
2. `transformer_block`
3. `rmsnorm`
4. `multihead_self_attention_with_rope`
5. `linear`
6. `split_heads`
7. `apply_rope`
8. `build_causal_mask`
9. `scaled_dot_product_attention`
10. `merge_heads`
11. `linear`（输出投影）
12. `rmsnorm`
13. `swiglu`
14. `silu`
15. `linear`（lm head）

下面按这个思路依次解释。

## 1. 输入层：`embedding()`

### 数学含义

设 token embedding 矩阵为：

$$
E \in \mathbb{R}^{V \times d_{model}}
$$

其中 `V` 是词表大小，`d_model` 是隐藏维度。对一个 token id `t`，embedding 就是取出第 `t` 行：

$$
\mathrm{embedding}(t) = E[t]
$$

如果输入是一整个 token id 张量 `token_ids`，那么输出就是在最后补上一维 `d_model`。

### 代码为什么这样写

代码是：

```python
return weights[token_ids]
```

原因很直接：PyTorch 允许直接用整型张量对矩阵做索引。于是：

- `weights` 形状是 `(vocab_size, d_model)`
- `token_ids` 形状可以是 `(...)`
- 索引结果自动变成 `(..., d_model)`

这其实就是“查表”，不是矩阵乘法。

### 复习时要记住

- embedding 的本质不是“算出来”，而是“取出来”。
- token id 是离散索引，所以最自然的实现就是直接索引 embedding 矩阵。

## 2. 线性变换基础：`linear()`

Transformer 里很多地方都在重复用线性变换：

- `Q, K, V` 投影
- attention 输出投影 `W_O`
- FFN / SwiGLU 里的 `W1, W2, W3`
- 最后的 `lm_head`

### 数学公式

如果按列向量写法，线性层通常写成：

$$
y = W x
$$

但这份代码里的输入张量习惯是把特征维放在最后，也就是把每个 token 表示看成“行向量”。所以代码实现写成：

$$
Y = X W^T
$$

### 代码为什么这样写

代码是：

```python
return in_features @ weights.T
```

其中：

- `weights` 形状固定是 `(d_out, d_in)`
- `in_features` 最后一维是 `d_in`
- `weights.T` 形状是 `(d_in, d_out)`

因此矩阵乘法结果的最后一维就会变成 `d_out`。

### 为什么这个 helper 很重要

因为后面所有投影都复用它。你复习时只要牢牢记住一件事：

> 这个项目里“线性层”的统一接口是：权重存成 `(d_out, d_in)`，真正算的时候用 `x @ W.T`。

## 3. 归一化与 FFN：`rmsnorm()`、`silu()`、`swiglu()`

### 3.1 `rmsnorm()`

RMSNorm 和 LayerNorm 很像，但它 **不减均值**，只做均方根缩放。

对单个向量 `x \in \mathbb{R}^{d}`，公式是：

$$
\mathrm{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \varepsilon}
$$

$$
\mathrm{RMSNorm}(x) = \frac{x}{\mathrm{RMS}(x)} \odot w
$$

其中 `w` 是可学习缩放参数。

代码里关键两步是：

```python
rms = torch.sqrt(in_temp.pow(2).mean(dim=-1, keepdim=True) + eps)
y = in_temp / rms * weights.to(torch.float32)
```

为什么这么写：

- `mean(dim=-1, keepdim=True)`：只沿最后一维做归一化，因为最后一维才是特征维。
- `keepdim=True`：方便后面广播除法，不会丢维度。
- 先转 `float32`：半精度下做平方、均值、开根号更容易有数值误差，所以先升精度更稳。
- 最后再转回原 dtype：既保留数值稳定性，又保持输出 dtype 和输入一致。

### 3.2 `silu()`

SiLU 的公式是：

$$
\mathrm{SiLU}(x) = x \cdot \sigma(x)
$$

代码：

```python
return in_features * torch.sigmoid(in_features)
```

为什么这样写：SiLU 是逐元素激活，所以只要把每个元素乘上它自己的 sigmoid 即可。

### 3.3 `swiglu()`

SwiGLU 是这份作业里的 FFN 结构，公式是：

$$
\mathrm{SwiGLU}(x) = W_2\big(\mathrm{SiLU}(W_1 x) \odot W_3 x\big)
$$

含义是：

1. 一路用 `W1` 做投影后过 SiLU
2. 另一路用 `W3` 做投影当 gate
3. 两路逐元素相乘
4. 再用 `W2` 投回 `d_model`

代码里正是：

```python
linear(w2_weight, silu(linear(w1_weight, in_features)) * linear(w3_weight, in_features))
```

复习时要记住：

- `W1` 和 `W3` 都是把 `d_model -> d_ff`
- `W2` 再把 `d_ff -> d_model`
- 中间的 `*` 不是矩阵乘法，而是逐元素门控。

## 4. 注意力打分前的形状准备：`split_heads()` 与 `merge_heads()`

多头注意力的核心思想是：把一个大向量拆成多个 head，每个 head 各自做注意力，再把结果拼回去。

### 4.1 `split_heads()`

输入形状是：

$$
(..., seq\_len, d_{model})
$$

如果有 `num_heads = h`，那么：

$$
d_{model} = h \cdot d_{head}
$$

第一步先 reshape：

$$
(..., seq\_len, d_{model}) \to (..., seq\_len, h, d_{head})
$$

第二步再交换 `seq_len` 和 `num_heads`：

$$
(..., seq\_len, h, d_{head}) \to (..., h, seq\_len, d_{head})
$$

代码里对应：

```python
x = x.reshape(*x.shape[:-1], num_heads, head_dim)
x = x.transpose(-3, -2)
```

为什么不是直接 `view` 成最终形状？

因为“拆维”和“调顺序”是两件事：

- `reshape` 负责把最后一维拆成 `(num_heads, head_dim)`
- `transpose` 负责把 head 维挪到 sequence 前面，方便后续 attention 逐 head 计算

### 4.2 `merge_heads()`

这是 `split_heads()` 的逆操作。

输入：

$$
(..., h, seq\_len, d_{head})
$$

先 transpose 回去：

$$
(..., h, seq\_len, d_{head}) \to (..., seq\_len, h, d_{head})
$$

再把最后两维合并：

$$
(..., seq\_len, h, d_{head}) \to (..., seq\_len, h \cdot d_{head})
$$

代码里是：

```python
x = x.transpose(-3, -2).contiguous()
x = x.reshape(*x.shape[:-2], num_heads * head_dim)
```

为什么常见实现会加 `.contiguous()`？

因为 `transpose` 之后内存布局可能不连续，先变连续再 `reshape` 更稳。

## 5. 因果注意力：`build_causal_mask()` 与 `scaled_dot_product_attention()`

### 5.1 `build_causal_mask()`

自回归语言模型要求当前位置只能看见“自己和过去”，不能看见未来。

所以 mask 必须满足：

$$
mask[i, j] =
\begin{cases}
\text{True}, & j \le i \\
\text{False}, & j > i
\end{cases}
$$

代码直接用下三角矩阵：

```python
torch.tril(torch.ones(sequence_length, sequence_length, dtype=torch.bool, device=device))
```

为什么这样写：

- `torch.ones(...)` 先造全 `True`
- `torch.tril(...)` 保留下三角
- `dtype=torch.bool` 表示这是逻辑 mask，不是数值权重

### 5.2 `scaled_dot_product_attention()`

这是 attention 的数学核心。给定：

- `Q`: `(..., q_len, d_k)`
- `K`: `(..., k_len, d_k)`
- `V`: `(..., k_len, d_v)`

公式是：

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V
$$

在这份代码里，mask 不是直接“加上一个矩阵”，而是先把非法位置填成 `-inf`，这样 softmax 后概率就会变成 0。

代码拆开看：

```python
att_logits = Q @ K.transpose(-2, -1)
att_logits = att_logits / (d_k ** 0.5)
att_logits = att_logits.masked_fill(~mask, float("-inf"))
att_weights = torch.softmax(att_logits, dim=-1)
output = att_weights @ V
```

### 为什么是这些具体写法

#### `K.transpose(-2, -1)`

因为 `Q` 和 `K` 往往不是 2 维，而是可能带 batch/head 维。我们只想交换最后两维：

$$
(..., k\_len, d_k) \to (..., d_k, k\_len)
$$

如果写成 `K.T`，在高维张量上会把所有维度反过来，不符合这里的需求。

#### 为什么除以 `\sqrt{d_k}`

因为点积的量级会随着维度 `d_k` 变大而增大。除以 `\sqrt{d_k}` 能让 logits 的尺度更稳定，softmax 不容易过于尖锐。

#### 为什么用 `masked_fill(~mask, -inf)`

题目里约定：

- `mask=True`：允许注意
- `mask=False`：禁止注意

所以要对 `~mask` 的位置填 `-inf`。

#### 为什么 softmax 后又再 `masked_fill(..., 0.0)` 一次

这是数值稳健性处理。极端情况下如果一整行都被 mask，softmax 可能出现数值问题；再次置零可以避免传播出脏值。

### 复习时要记住

- `QK^T` 算的是“每个 query 对每个 key 的相关性”。
- softmax 是在最后一维做，也就是“对每个 query，在所有 key 上归一化”。
- 最后乘 `V` 的本质是：用这些注意力权重对 value 做加权求和。

## 6. 位置编码：`apply_rope()`

RoPE（Rotary Position Embedding）不是把位置向量“加到”表示上，而是把 `Q` 和 `K` 的每一对维度按位置做旋转。

### 数学公式

把向量最后一维两两配对：

$$
(x_0, x_1), (x_2, x_3), \dots
$$

对第 `i` 对维度，频率定义为：

$$
\omega_i = \theta^{-2i/d_k}
$$

对位置 `p`，旋转角度是：

$$
\varphi_{p, i} = p \cdot \omega_i
$$

然后二维旋转矩阵作用在每一对维度上：

$$
\begin{bmatrix}
x'_{2i} \\
x'_{2i+1}
\end{bmatrix}
=
\begin{bmatrix}
\cos \varphi_{p,i} & -\sin \varphi_{p,i} \\
\sin \varphi_{p,i} & \cos \varphi_{p,i}
\end{bmatrix}
\begin{bmatrix}
x_{2i} \\
x_{2i+1}
\end{bmatrix}
$$

### 代码和公式的对应关系

#### 1. 先检查维度

最后一维必须能两两成对，所以 `d_k` 必须是偶数。代码里：

```python
if d_k % 2 != 0:
    raise ValueError(...)
```

#### 2. 处理 `token_positions`

`token_positions` 的最后一维必须和 `seq_len` 对齐，因为每个 token 都要有自己的位置编号。

接着通过循环 `unsqueeze(-2)`，把它扩成能和 `(..., num_heads, seq_len, head_dim)` 广播兼容的形状。这一步非常关键，因为 RoPE 在这个项目里要支持任意数量的前导维度，不只是 `(batch, seq_len, d_k)`。

#### 3. 计算频率和角度

代码：

```python
i = torch.arange(half_dim, device=x.device, dtype=compute_dtype)
inv_freq = theta ** (-2.0 * i / d_k)
angles = positions * inv_freq
```

这正对应上面的：

$$
\omega_i = \theta^{-2i/d_k}
$$

和

$$
\varphi_{p,i} = p \cdot \omega_i
$$

#### 4. 奇偶拆分并旋转

代码：

```python
x_even = x[..., 0::2]
x_odd = x[..., 1::2]

out_even = x_even * cos_angles - x_odd * sin_angles
out_odd = x_even * sin_angles + x_odd * cos_angles
```

这就是标准二维旋转公式。

#### 5. 为什么最后用 `torch.stack(...).flatten(-2)`

因为旋转后我们手里有两组张量：

- 偶数位结果 `out_even`
- 奇数位结果 `out_odd`

先 `stack` 成 `(..., half_dim, 2)`，再 `flatten(-2)`，就能恢复成原来的最后一维 `d_k`。

### 为什么 RoPE 只作用在 `Q` 和 `K`

因为 attention 的相对位置关系体现在 `Q` 和 `K` 的相似度上，也就是 `QK^T` 这一项。`V` 承载的是被聚合的信息本身，不需要做同样的旋转。

## 7. 多头自注意力：`multihead_self_attention()` 与 `multihead_self_attention_with_rope()`

### 7.1 `multihead_self_attention()`

这是普通版的因果多头自注意力，不含 RoPE。整体流程是：

$$
X \xrightarrow{W_Q, W_K, W_V} Q, K, V
\xrightarrow{split\ heads}
\xrightarrow{causal\ attention}
\xrightarrow{merge\ heads}
\xrightarrow{W_O}
Y
$$

更细一点：

1. `Q = linear(q_proj_weight, in_features)`
2. `K = linear(k_proj_weight, in_features)`
3. `V = linear(v_proj_weight, in_features)`
4. 分别 `split_heads`
5. 用 `build_causal_mask(seq_len, device)` 生成下三角 mask
6. 用 `scaled_dot_product_attention(Q, K, V, mask)` 做逐 head 注意力
7. `merge_heads`
8. `linear(o_proj_weight, attn_out)` 做输出投影

### 为什么最后还要有一次 `W_O`

因为 `merge_heads()` 只是把多个 head 的结果拼回去，它只负责形状恢复，不负责让不同 head 的信息重新线性混合。真正完成这一步的是输出投影 `W_O`。

### 7.2 `multihead_self_attention_with_rope()`

这和普通版几乎完全一样，唯一差别是：

- 在 `split_heads()` 之后
- 在进入 attention 之前
- 对 `Q` 和 `K` 额外做一次 `apply_rope()`

也就是：

$$
Q, K, V
\xrightarrow{split\ heads}
\tilde Q = \mathrm{RoPE}(Q),\quad \tilde K = \mathrm{RoPE}(K)
$$

然后再做：

$$
\mathrm{Attention}(\tilde Q, \tilde K, V)
$$

### 为什么 RoPE 要在 `split_heads()` 之后做

因为这个项目里的 RoPE 维度应该是 **每个 head 的维度**，也就是：

$$
d_{head} = d_{model} / num\_heads
$$

如果在 split 之前做 RoPE，就会把整个 `d_model` 当作一个旋转空间，和多头注意力的定义不一致。

### 为什么 `token_positions is None` 时要构造 `0..seq_len-1`

因为在最普通的自注意力场景里，token 就按顺序排在序列中，第 0 个 token 的位置就是 0，第 1 个就是 1，依此类推。只有做更复杂的场景时，才需要外部传入自定义位置。

## 8. 单层 Block：`transformer_block()`

这份代码采用的是 **pre-norm Transformer block**。也就是说：先归一化，再做 attention / FFN，然后加残差。

### 数学结构

给输入 `X`，第一部分是注意力子层：

$$
Y = X + \mathrm{MHA\_RoPE}(\mathrm{RMSNorm}(X))
$$

第二部分是前馈网络子层：

$$
Z = Y + \mathrm{SwiGLU}(\mathrm{RMSNorm}(Y))
$$

最终返回 `Z`。

### 代码层面的拆解

```python
x = in_features
attn_input = rmsnorm(x, ln1_weight)
attn_output = multihead_self_attention_with_rope(...)
x = x + attn_output

ffn_input = rmsnorm(x, ln2_weight)
ffn_output = swiglu(ffn_input, w1_weight, w2_weight, w3_weight)
x = x + ffn_output
return x
```

### 为什么 residual 要这样放

残差连接的意义是：

- 让网络更容易优化
- 保留原输入信息
- 避免深层网络训练时梯度过弱

所以不是把 attention 输出直接覆盖 `x`，而是 `x + attn_output`。

### 为什么这里用的是 `multihead_self_attention_with_rope()`

因为这份模型最终是带 RoPE 的语言模型，所以 block 里的 attention 子层也必须是带 RoPE 的版本。普通 `multihead_self_attention()` 更像一个不带位置旋转的基础版 helper。

## 9. 整个语言模型：`transformer_lm()`

这是整份文件里最顶层的函数，它把前面所有零件真正拼成了一个语言模型。

### 数学流程

对输入 token id 序列 `T`：

1. 先查 embedding：

$$
X_0 = \mathrm{Embedding}(T)
$$

2. 依次过 `L` 个 block：

$$
X_{\ell+1} = \mathrm{TransformerBlock}_\ell(X_\ell)
$$

3. 最后做一次 RMSNorm：

$$
\hat X_L = \mathrm{RMSNorm}(X_L)
$$

4. 再做 lm head 投影得到 logits：

$$
\mathrm{logits} = \hat X_L W_{lm}^T
$$

### 代码为什么这样写

代码的核心结构是：

```python
x = embedding(token_embedding_weight, in_indices)
for block in block_weights:
    x = transformer_block(x, **block, num_heads=num_heads, theta=theta, max_seq_len=max_seq_len)
x = rmsnorm(x, ln_final_weight)
logits = linear(lm_head_weight, x)
return logits
```

为什么这段写法非常自然：

- `embedding(...)`：把离散 token id 变成连续向量
- `for block in block_weights`：每一层结构一样，只是参数不同，所以循环最自然
- `**block`：因为每层 block 的权重已经整理成和 `transformer_block()` 参数名一致的字典
- `rmsnorm(...)`：最后一层 block 后面还要再做一次 final norm
- `linear(lm_head_weight, x)`：把最后隐藏状态投到词表大小，得到每个位置上对所有词的打分

### 为什么返回的是 logits，不是概率

因为训练语言模型时，通常把 logits 直接送进交叉熵损失。交叉熵内部会自己做 softmax，所以这里不应该提前做 softmax。

### 复习时的总总结

如果只用一句话概括整份 `model.py`，可以记成：

> 这份文件是在用最基础的 PyTorch 张量操作，把一个带 RoPE、带 RMSNorm、带 SwiGLU 的 pre-norm Transformer language model 从零拼出来。

复习的时候，最好始终沿着这条链回忆：

`token ids -> embedding -> block x N -> final rmsnorm -> lm head -> logits`

而每个 block 内部再拆成：

`rmsnorm -> qkv linear -> split heads -> rope -> causal attention -> merge heads -> output linear -> residual -> rmsnorm -> swiglu -> residual`

## 10. 速查表：每个函数到底在做什么

| 函数 | 一句话作用 | 核心公式/动作 |
| --- | --- | --- |
| `linear` | 对最后一维做无 bias 线性变换 | `x @ W.T` |
| `embedding` | 用 token id 查表得到向量 | `E[token_ids]` |
| `silu` | 逐元素激活 | `x * sigmoid(x)` |
| `rmsnorm` | 只按 RMS 缩放，不减均值 | `x / rms(x) * w` |
| `swiglu` | FFN 的门控结构 | `W2(SiLU(W1x) * W3x)` |
| `build_causal_mask` | 构造自回归下三角 mask | `torch.tril(...)` |
| `scaled_dot_product_attention` | 真正做注意力打分和聚合 | `softmax(QK^T / sqrt(d_k))V` |
| `split_heads` | 把 `d_model` 拆成多个 head | `(..., S, D) -> (..., H, S, d)` |
| `merge_heads` | 把多个 head 拼回去 | `(..., H, S, d) -> (..., S, D)` |
| `apply_rope` | 对 `Q/K` 做按位置旋转 | 每两维做 2D rotation |
| `multihead_self_attention` | 不带 RoPE 的因果多头注意力 | `QKV -> attention -> W_O` |
| `multihead_self_attention_with_rope` | 带 RoPE 的因果多头注意力 | `RoPE(Q), RoPE(K)` 后再 attention |
| `transformer_block` | 单层 pre-norm Transformer block | `attn residual + ffn residual` |
| `transformer_lm` | 整个语言模型前向 | `embedding -> blocks -> norm -> lm head` |

如果以后复习忘了细节，优先回忆三层结构：

1. 顶层：`transformer_lm`
2. 中层：`transformer_block`
3. 底层：attention / norm / FFN 的各个 helper

这样你就不会被很多小函数打散，而是始终知道每个函数在整台模型机器里的位置。